# 第80课 · 向量相似度检索 — 余弦相似度（cosine similarity） vs 点积 vs L2，纯 NumPy k-NN 实现

**目标**：用 L2 归一化 embedding 做余弦相似度搜索，找最相似的 k 首歌。

🔗 **Aurora 连接**：本节实现的 `find_similar()` 将成为 `aurora.music.recommend` 的底层检索引擎，Month 5 接入 FAISS 时只需替换这个函数。

← **上一课**　[L79 · 音乐嵌入向量](L79_music_embed.ipynb)

> 上节课学习了 **音乐嵌入向量**：对比学习：相似风格靠近，不同流派拉远。  
> 本课将探讨 **向量相似度检索**。

## 本课剧情：Spotify 的"你可能也喜欢"是怎么算出来的？

每首歌经过上节课的 MusicEncoder，变成一个128维向量。现在 Spotify 有 1 亿首歌，用户播放了一首 The Beatles，系统要在 0.1 秒内找出最相似的 10 首推荐。

**关键问题**：怎么衡量两个向量"有多相似"？

**三种距离，三种直觉**：

| 指标 | 公式 | 衡量的是… | 适用场景 |
|---|---|---|---|
| 余弦相似度 | cos(a,b) = a·b / (‖a‖·‖b‖) | 方向夹角 | 长短不重要，只看风格 |
| L2距离 | ‖a-b‖ | 空间距离 | 绝对位置重要 |
| 点积 | a·b | 方向+模长 | L2归一化后 = 余弦 |

**重要结论**：L2 归一化后，**点积 = 余弦相似度**。训练好的 embedding 通常做 L2 归一化，所以点积检索 = 余弦检索，且矩阵乘法可以批量高效计算。

**k-NN 检索的朴素实现**：计算 query 与所有 N 个向量的相似度，取 top-k。
- 复杂度：O(N×d)，N=1亿时太慢
- 近似最近邻（FAISS）：用索引压缩，O(√N)，损失少量精度

**本节任务**：用纯 NumPy 实现 `find_similar()`，验证余弦相似度检索的正确性。

In [ ]:
import numpy as np
import time

## 1. 余弦相似度与单位向量

余弦相似度定义：

```
cos_sim(a, b) = dot(a, b) / (||a|| * ||b||)
```

值域 `[-1, 1]`：1 表示方向完全相同（最相似），-1 表示方向相反，0 表示正交（无关）。

**关键技巧**：如果先把所有向量 L2 归一化（变成单位向量，||v|| = 1），则 `cos_sim(a, b) = dot(a, b)`，矩阵乘法一步搞定，不需要每次除以模长。

## 1b. L2 距离（欧氏距离）：与余弦相似度的对比

L2 距离（欧氏距离）定义：

```
L2(a, b) = ||a - b|| = sqrt(sum((a_i - b_i)^2))
```

值域 `[0, +∞)`：0 表示完全相同，越大表示越不相似。

**余弦相似度 vs L2 距离的本质区别**：

| 度量 | 关注什么 | 适合场景 |
|------|----------|----------|
| 余弦相似度 | 向量**方向**（忽略长度） | 文本/音频 embedding，方向代表语义 |
| L2 距离 | 向量**绝对距离**（方向+长度） | 像素距离、物理坐标、推荐分数差异 |

**关键联系**：若向量已 L2 归一化（||v|| = 1），则：

```
L2(a, b)^2 = 2 - 2 * cos_sim(a, b)
```

即归一化后，最大化余弦相似度等价于最小化 L2 距离。
本节选择余弦相似度路线，因为 embedding 模长通常无意义（只有方向携带语义）。


In [ ]:
# 演示：归一化后余弦相似度与 L2 距离的等价性
a = np.array([3.0, 4.0])   # 模长 5
b = np.array([1.0, 0.0])   # 模长 1

# L2 距离（原始向量）
l2_raw = np.linalg.norm(a - b)

# 归一化后验证等价关系
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
cos_sim_val = np.dot(a_n, b_n)
l2_normalized = np.linalg.norm(a_n - b_n)
l2_from_cos = np.sqrt(2 - 2 * cos_sim_val)

print(f"L2(a,b) raw         = {l2_raw:.4f}")
print(f"L2(a_n, b_n)        = {l2_normalized:.4f}")
print(f"sqrt(2-2*cos_sim)   = {l2_from_cos:.4f}")
assert np.isclose(l2_normalized, l2_from_cos), "等价关系验证失败"
print("\n✅ 验证：归一化后 L2^2 = 2 - 2*cos_sim")
print(f"   cos_sim={cos_sim_val:.4f}  L2_norm^2={l2_normalized**2:.4f}  2-2*cos={2-2*cos_sim_val:.4f}")


In [ ]:
# 演示：L2 归一化后点积 == 余弦相似度
a = np.array([3.0, 4.0])   # 模长 5
b = np.array([1.0, 0.0])   # 模长 1

# 原始余弦相似度
cos_raw = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 归一化后点积
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
cos_normed = np.dot(a_n, b_n)

print(f'原始余弦相似度: {cos_raw:.4f}')
print(f'归一化后点积:   {cos_normed:.4f}')
print(f'相等: {np.isclose(cos_raw, cos_normed)}')

## 2. 暴力 top-k：argpartition vs argsort

从 N 个分数中找最大的 k 个，有两种做法：

```
argsort:        O(N log N)  — 完整排序，返回全部排名
argpartition:   O(N)        — 只保证 top-k 进入最后 k 个槽，内部不排序
```

当 N 很大（百万首歌）而 k 很小（返回 5 首）时，`argpartition` 明显更快。用法：`np.argpartition(scores, -k)[-k:]` 拿到 top-k 的索引（顺序随机），再用 `scores[idx].argsort()[::-1]` 对这 k 个排序。

In [ ]:
# 演示：argpartition vs argsort 速度对比
rng = np.random.default_rng(42)
N, k = 100_000, 5
scores = rng.random(N)

t0 = time.perf_counter()
for _ in range(200):
    idx_sort = np.argsort(scores)[-k:][::-1]
t_sort = (time.perf_counter() - t0) / 200 * 1e6

t0 = time.perf_counter()
for _ in range(200):
    idx_part = np.argpartition(scores, -k)[-k:]
    idx_part = idx_part[np.argsort(scores[idx_part])[::-1]]
t_part = (time.perf_counter() - t0) / 200 * 1e6

print(f'argsort      平均: {t_sort:.1f} µs')
print(f'argpartition 平均: {t_part:.1f} µs')
print(f'结果一致: {np.array_equal(idx_sort, idx_part)}')

## 3. FAISS：大规模近似最近邻（Approximate Nearest Neighbor，ANN）

[FAISS](https://github.com/facebookresearch/faiss) 是 Meta 开发的向量检索库，支持 IVF、HNSW 等索引结构，在百万量级下仍能毫秒级返回结果。

本节先用暴力 `np.dot` 实现，接口固定为 `(indices, scores)`。Month 5 只需把函数体换成 FAISS `index.search()`，上层代码零改动。

```
暴力检索（本节）:  适合 N < 10 万，无需安装额外依赖
FAISS（Month 5）:  适合 N > 10 万，需 pip install faiss-cpu
```

In [ ]:
# 演示：FAISS 接口预览（Month 5 才真正安装，这里只看调用形态）
print('暴力检索接口:')
print('  indices, scores = find_similar(query_emb, library_embs, top_k=5)')
print()
print('FAISS 接口（Month 5）:')
print('  import faiss')
print('  index = faiss.IndexFlatIP(d)  # 内积 = 余弦（已归一化）')
print('  index.add(library_embs)')
print('  scores, indices = index.search(query_emb[None], top_k)')

## 4. ✏️ 实现 `find_similar(query_emb, library_embs, top_k=5)`

**签名**：
```python
def find_similar(
    query_emb: np.ndarray,          # (d,) 查询向量
    library_embs: np.ndarray,       # (N, d) 歌曲库
    top_k: int = 5
) -> tuple[np.ndarray, np.ndarray]:
    # 返回: (indices, scores)
    # indices: (top_k,) int，从高到低排序的索引
    # scores: (top_k,) float，对应余弦相似度
```

**4步实现路线**：

| 步骤 | 操作 | NumPy 工具 |
|---|---|---|
| 1 | L2 归一化 query | `q = query_emb / np.linalg.norm(query_emb)` |
| 2 | L2 归一化 library（逐行） | `lib = library_embs / np.linalg.norm(library_embs, axis=1, keepdims=True)` |
| 3 | 批量点积（= 余弦相似度） | `scores = lib @ q` → shape (N,) |
| 4 | top-k：`argpartition` + `argsort` | `np.argpartition(-scores, top_k)[:top_k]` |

**验收标准**：
- 正交 embedding（`np.eye(4)`）：query[0] 的最近邻是自己（score=1.0）
- 相似度按降序排列
- 返回 indices 和 scores，shape 均为 (top_k,)

In [ ]:
def find_similar(
    query_emb: np.ndarray,
    library_embs: np.ndarray,
    top_k: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """余弦相似度 top-k 检索。

    参数
    ----
    query_emb   : shape (d,)   查询向量
    library_embs: shape (N, d) 歌曲库 embedding
    top_k       : 返回最相似的 k 首

    返回
    ----
    indices : shape (top_k,)  在 library_embs 中的行索引，按相似度降序
    scores  : shape (top_k,)  对应余弦相似度
    """
    # ✏️ TODO 1: L2 归一化 query_emb -> q
    # q = ...

    # ✏️ TODO 2: L2 归一化 library_embs 每行 -> L
    # L = ...

    # ✏️ TODO 3: scores = L @ q，shape (N,)
    # scores = ...

    # ✏️ TODO 4: argpartition 取 top_k 候选，再降序排列
    # top_k = min(top_k, len(scores))
    # part_idx = ...
    # order = ...
    # indices = ...

    raise NotImplementedError("请实现 find_similar")

In [ ]:
# 基础正确性检查
embs = np.eye(4)  # 4 首歌，各自正交
try:
    result = find_similar(embs[0], embs, top_k=2)
except NotImplementedError:
    result = None

if result is None or result is ...:
    print("⬜ find_similar 未实现")
else:
    idx, s = result
    assert 0 in idx, f'index 0（完全匹配）必须在 top-2 中，得到 {idx}'
    assert np.isclose(s[0], 1.0), f'最高分应为 1.0，得到 {s[0]:.4f}'
    assert s[0] >= s[1], '分数应降序排列'
    print('✅ 基础检查通过')

    # 返回形状检查
    rng = np.random.default_rng(0)
    big_lib = rng.standard_normal((100, 32))
    q = rng.standard_normal(32)
    idx2, s2 = find_similar(q, big_lib, top_k=10)
    assert idx2.shape == (10,), f'indices shape 应为 (10,)，得到 {idx2.shape}'
    assert s2.shape == (10,), f'scores shape 应为 (10,)，得到 {s2.shape}'
    assert all(s2[i] >= s2[i+1] for i in range(9)), '分数未降序'
    print('✅ 形状与降序检查通过')

## 5. 参数实验：50首歌库，查询10次

**参数**：`N=50`（歌曲数），`d=128`（embedding 维度），`top_k=5`

**预期现象**：
- 每次检索时间 < 1ms（暴力 dot 在 N=50 量级极快）
- `scores` 范围在 `[-1, 1]`，最高分接近 1（若 query 来自库中）
- 调大 `top_k` 到 50 结果仍正确（等于返回全部歌曲排名）

In [ ]:
rng = np.random.default_rng(7)
N, d, top_k = 50, 128, 5
library = rng.standard_normal((N, d))

times = []
for i in range(10):
    query = library[i]  # 用库中第 i 首歌作为 query，期望自身得分最高
    t0 = time.perf_counter()
    try:
        result = find_similar(query, library, top_k=top_k)
    except NotImplementedError:
        print("⬜ find_similar 未实现"); break
    if result is None:
        print("⬜ find_similar 未实现"); break
    idx, scores = result
    elapsed_us = (time.perf_counter() - t0) * 1e6
    times.append(elapsed_us)
    top1_is_self = idx[0] == i
    print(f'query={i:2d}  top1={idx[0]:2d}(自身={top1_is_self})  '
          f'score={scores[0]:.4f}  时间={elapsed_us:.1f}µs')

if not times:
    print('⬜ find_similar 未实现，跳过性能断言')
else:
    avg_ms = np.mean(times) / 1000
    print(f'\n平均检索时间: {avg_ms:.3f} ms  (应 < 1 ms for N={N})')
    assert avg_ms < 1.0, f'检索太慢：{avg_ms:.3f} ms'

## 本课收束

`find_similar()` 接受一个查询 embedding 和歌曲库矩阵，输出按余弦相似度降序排列的 `(indices, scores)` 二元组。它将作为 `aurora.music.recommend` 的底层检索引擎，上层只需传入 encoder 生成的 embedding，无需了解检索细节。Month 5 引入 FAISS 后，只需替换此函数内部实现，接口保持不变——下一节 L81 将在此基础上构建完整的播放列表推荐流水线。

## ✏️ 白板挑战：相似度检索手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：a=[3,4]，b=[1,0]，余弦相似度=？L2距离=？  
（cos=a·b/(‖a‖·‖b‖)=(3)/(5×1)=0.6；L2=‖[2,4]‖=√20≈4.47）

**问 2**：L2归一化后a'=[0.6,0.8]，b'=[1,0]，验证：余弦相似度=点积？  
（a'·b'=0.6×1+0.8×0=0.6；cos(a',b')=a'·b'/(‖a'‖·‖b'‖)=0.6/(1×1)=0.6 ✓）

**问 3**：N=1亿，d=128，暴力k-NN每次查询需要多少次乘法？  
（N×d=10^8×128=1.28×10^10次；若每次1ns则需12.8秒，太慢）

**问 4**：`argpartition` vs `argsort` 找 top-5：时间复杂度各是多少？  
（argpartition: O(N)，只保证top-k正确但不排序；argsort: O(N log N)，完全排序；大N时argpartition更快）

**问 5**：若 query 向量全为零（零向量），L2 归一化会发生什么？应该如何处理？  
（0/‖0‖=0/0=nan；应检查norm>1e-8再归一化，或返回随机结果/空列表）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：余弦相似度和L2距离
a_q1 = np.array([3.0, 4.0])
b_q1 = np.array([1.0, 0.0])
cos_q1 = np.dot(a_q1, b_q1) / (np.linalg.norm(a_q1) * np.linalg.norm(b_q1))
l2_q1 = np.linalg.norm(a_q1 - b_q1)
assert abs(cos_q1 - 0.6) < 1e-6, f"cos应=0.6，得{cos_q1:.4f}"
assert abs(l2_q1 - np.sqrt(20)) < 1e-6, f"L2应=√20，得{l2_q1:.4f}"
print(f"Q1 ✅  cos([3,4],[1,0])={cos_q1:.4f}，L2={l2_q1:.4f}（=√20≈{np.sqrt(20):.4f}）")

# 问2：L2归一化后点积=余弦
a_n = a_q1 / np.linalg.norm(a_q1)
b_n = b_q1 / np.linalg.norm(b_q1)
dot_normalized = np.dot(a_n, b_n)
cos_normalized = np.dot(a_n, b_n) / (np.linalg.norm(a_n) * np.linalg.norm(b_n))
assert abs(dot_normalized - cos_normalized) < 1e-9, "归一化后点积应=余弦"
assert abs(dot_normalized - 0.6) < 1e-6, f"应=0.6，得{dot_normalized:.4f}"
print(f"Q2 ✅  归一化后: 点积={dot_normalized:.4f}，余弦={cos_normalized:.4f}（完全相等）")

# 问3：暴力k-NN乘法次数
N_q3, d_q3 = 10**8, 128
ops_q3 = N_q3 * d_q3
print(f"Q3 ✅  N=1亿,d=128: 乘法次数={ops_q3:.2e}（每次1ns需{ops_q3*1e-9:.1f}秒，太慢→需要FAISS）")

# 问4：argpartition vs argsort速度对比
rng_q4 = np.random.default_rng(42)
scores_q4 = rng_q4.standard_normal(100000)
k_q4 = 5
# argpartition: O(N)
idx_part = np.argpartition(-scores_q4, k_q4)[:k_q4]
top_part = scores_q4[idx_part]
# argsort: O(N log N)
idx_sort = np.argsort(-scores_q4)[:k_q4]
top_sort = scores_q4[idx_sort]
assert set(idx_part) == set(idx_sort[:k_q4]), "两种方法结果集合应相同"
print(f"Q4 ✅  argpartition返回集合={sorted(scores_q4[idx_part])[::-1][:2]}...，argsort结果={sorted(scores_q4[idx_sort])[::-1][:2]}...（集合相同）")

# 问5：零向量处理
# 零向量 L2 归一化 = 0/0 = nan；参考实现在分母加 1e-12 避免 nan，
# 使得零向量 query 仍能安全返回（相似度全为 0，而非崩溃）。
try:
    zero_query = np.zeros(128)
    norm_val = np.linalg.norm(zero_query)
    if norm_val < 1e-8:
        print(f"Q5 ✅  零向量norm={norm_val:.2e}<1e-8，朴素归一化会得 nan（需加 eps 或跳过）")
    else:
        print(f"Q5 ✅  向量norm={norm_val:.4f}，正常归一化")
    # 用维度匹配的零向量查询库（query 与 library 同为 4 维），验证安全返回
    zq = np.zeros(4)
    idx_z, sc_z = find_similar(zq, np.eye(4), top_k=2)
    assert not np.isnan(sc_z).any(), "零向量不应产生 nan 分数"
    print(f"Q5 ✅  find_similar(零向量)安全返回: indices={idx_z}，scores[0]={sc_z[0]:.4f}（无 nan）")
except NotImplementedError:
    print(f"Q5 ✅  零向量→nan；应在归一化前加 eps 或检查 norm>1e-8（待实现后验证）")

print("\n🎉 相似度检索白板挑战通过！")


In [ ]:
# ✏️ 本课自评
l80_review = {
    "cosine_vs_l2_understood":  None,  # 理解cos/L2/点积的差异，L2归一化后点积=余弦？True/False
    "topk_complexity":          None,  # 理解argpartition O(N) vs argsort O(NlogN)？True/False
    "find_similar_impl":        None,  # find_similar()实现正确，正交embedding→自身相似度=1？True/False
    "zero_vector_edge":         None,  # 理解零向量归一化问题，知道如何处理？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l80_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l80_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L80 全部通关！进入 L81：音乐推荐系统')

---

→ **下一课**　[L81 · 音乐推荐系统](L81_recommendation.ipynb)

> 下节课将学习 **音乐推荐系统**：用户喜好 → 嵌入向量 → k-NN 邻居 → 推荐列表。